# Colab → SSH access for the distillation run

Turns this Colab T4 instance into an SSH server reachable from the OpenHands
agent over a **cloudflared Quick Tunnel** (fallback path, spec §3.4).

**Credentials:** key-only, `PasswordAuthentication no`. No password is ever
set, printed, or stored. The public key below is the only credential.

**Workflow:** run all cells → the last cell prints the `trycloudflare.com`
hostname. Hand that hostname + the private key to the agent; the agent
connects with `ssh -p 2222 <tunnel-host>` via cloudflared and runs the
full `scripts/distill/*` pipeline on the GPU.

In [ ]:
# 1. Install ssh server + cloudflared tunnel client
!apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get install -qq -y openssh-server cloudflared > /dev/null 2>&1
print('ok: sshd + cloudflared installed')

In [ ]:
# 2. Configure sshd — KEY-ONLY, no password, root allowed (Colab runs as root)
import os

PUBKEY = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAICiDCgsb4wDRjs1DQ4nMwYclYhNJDX7qDP7FzT5kcLq6 openhands-colab-auth"

os.makedirs('/root/.ssh', exist_ok=True)
with open('/root/.ssh/authorized_keys', 'w') as f:
    f.write(PUBKEY + '\n')

# drop-in, merge-safe sshd config
conf = '''\
Port 2222
PermitRootLogin prohibit-password
PasswordAuthentication no
ChallengeResponseAuthentication no
UsePAM no
PubkeyAuthentication yes
AuthorizedKeysFile  /root/.ssh/authorized_keys
'''
with open('/etc/ssh/sshd_config.d/99-colab.conf', 'w') as f:
    f.write(conf)
print('sshd: key-only on :2222, root via key')
print('authorized_keys entries:', len(open('/root/.ssh/authorized_keys').readlines()))

In [ ]:
# 3. Start sshd
!service ssh start || /usr/sbin/sshd
!sleep 1; (ss -tlnp | grep 2222) || echo 'port 2222 not yet listening (check /var/log/auth.log)'
print('sshd started')

In [ ]:
# 4. Start cloudflared tunnel -> local ssh :2222
import subprocess, re, time
log = open('/tmp/cloudflared.log', 'w')
p = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'ssh://localhost:2222'],
                     stdout=log, stderr=subprocess.STDOUT)
host = None
for _ in range(60):
    time.sleep(1)
    data = open('/tmp/cloudflared.log').read()
    if 'trycloudflare.com' in data:
        m = re.search(r'https://([0-9a-z-]+)\.trycloudflare\.com', data)
        if m:
            host = m.group(1) + '.trycloudflare.com'
            break
print('TUNNEL_HOST=' + (host or 'waiting… see /tmp/cloudflared.log'))

In [ ]:
# 5. Clone repo so the agent can run scripts/distill immediately
import os, subprocess
REPO = '/content/remove-photo-bg'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/octoborg-x/remove-photo-bg.git', REPO], check=True)
print('repo:', REPO)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 6. Print the connect info (public info only)
import re
try:
    data = open('/tmp/cloudflared.log').read()
    m = re.search(r'https://([0-9a-z-]+)\.trycloudflare\.com', data)
    host = m.group(1) + '.trycloudflare.com'
except Exception:
    host = None

if host:
    print('PASTE THIS TO THE AGENT:')
    print(f'  TUNNEL_HOST={host}')
else:
    print('tunnel not ready — read /tmp/cloudflared.log or wait a moment and re-run')

# the private key is NOT here. it lives in the agent's ~/.ssh/